How are hotspots of house sparrows (houspa) in Berlin influenced by increased pollution in the years  2023-2025?(April-June)

Used libaries:

In [2]:
import time
import requests
import pandas as pd
from datetime import date, datetime, timedelta, timezone

Request hotspots in Berlin:

In [ ]:
base_url = "https://api.ebird.org/v2"

headers = {"X-eBirdApiToken": eBird_Key}

url = base_url + "/ref/hotspot/DE-BE"

response = requests.get(url, headers=headers)

if response.status_code == 200:
    hotspot_data = response.json()
    hotspot_df = pd.DataFrame(hotspot_data)    

Collecting the dates:

In [4]:
# Generating a list of relevant dates
def date_range_list(start, end):
    days = []
    cur = start
    while cur <= end:
        days.append(cur)
        cur += timedelta(days=1)
    return days

all_days = []

# Collecting Dates from 2023-2025
for y in [2023, 2024, 2025]:
    all_days.extend(list(date_range_list(date(y, 1, 1), date(y, 12, 31))))

API Request and function for getting the Houspa observation data 

In [5]:
def get_houspa_hotspot_counts(day):
    
    y = str(day.year)
    m = str(day.month).zfill(2)
    d = str(day.day).zfill(2)
    
    url = base_url + "/data/obs/DE-BE/historic/" + y + "/" + m + "/" + d
    
    response = requests.get(url, headers=headers, params={"speciesCode": "houspa"})
    
    species_observation = response.json()
    species_observation_df = pd.DataFrame(species_observation)
    
    # Filter observation data to only known hotspot locations 
    species_observation_hotspot_df = species_observation_df[species_observation_df["locId"].isin(hotspot_df["locId"])]
    
    # §LLM Help:  for the sum of observations if available
    if "howMany" in species_observation_hotspot_df.columns:
        how_many_sum = pd.to_numeric(species_observation_hotspot_df["howMany"], errors="coerce").sum()
        if pd.isna(how_many_sum):
            how_many_sum = 0
    else:
        how_many_sum = 0
    
    return {
        "date": str(day),
        "status": 200,
        "observation_count": int(len(species_observation_hotspot_df)),
        "sum_howMany_birds": float(how_many_sum)
    }  

Getting the Houspa observation data for 2023-2025:

In [ ]:
bird_results = []

# Getting the sum of Houspa observations in Berlin hotspots for the relevant dates 
for day in all_days:
    result = get_houspa_hotspot_counts(day)
    bird_results.append(result)
    
    time.sleep(0.2)  

count_bird_df = pd.DataFrame(bird_results)

API Request and function for getting pollution data:

In [15]:
# Berlin Coordinates
lat = 52.5200
lon = 13.4050

# Function for Getting Pollution data
def get_pollution_daily(day):

    # §LLM Help: for syntax of start and end time
    start_dt = datetime(day.year, day.month, day.day, tzinfo=timezone.utc)
    end_dt = start_dt + timedelta(days=1)

    url = "http://api.openweathermap.org/data/2.5/air_pollution/history"

    params = {
        "lat": lat,
        "lon": lon,
        "start": int(start_dt.timestamp()),
        "end": int(end_dt.timestamp()),
        "appid": OpenWeather_Key
    }

    response = requests.get(url, params=params)
    data = response.json()

    values = data["list"]
    pm2_5 = []
    no2 = []

    # Collecting all pollution data for the day
    for i in values:
        pm2_5.append(i["components"]["pm2_5"])
        no2.append(i["components"]["no2"])

    # Return daily mean value
    return {
        "date": str(day),
        "pm2_5_mean": sum(pm2_5) / len(pm2_5),
        "no2_mean": sum(no2) / len(no2)
    }

Getting polution data for the relevant dates

In [ ]:
results = []

# Getting pollution data for all days
for day in all_days:
    res = get_pollution_daily(day)
    results.append(res)
    
    time.sleep(0.5)

pollution_df = pd.DataFrame(results)

Merging bird an pollution data:

In [ ]:
count_bird_df["date"] = pd.to_datetime(count_bird_df["date"])
pollution_df["date"] = pd.to_datetime(pollution_df["date"])

# Merging Bird an Pollution data on date
combined_df = pd.merge(
    count_bird_df,
    pollution_df,
    on="date"
)

Create CSV File:

In [19]:
combined_df.to_csv("rq_2_berlin_houspa_pollution_2023_2025.csv", index=False)

Adding Columns for differences:

In [ ]:
df = pd.read_csv("rq_2_berlin_houspa_pollution_2023_2025.csv")

df["date"] = pd.to_datetime(df["date"])

# Calculating the difference to the day before
df["diff_howMany"] = df["sum_howMany_birds"].diff()
df["diff_pm2_5_mean"] = df["pm2_5_mean"].diff()
df["diff_no2_mean"] = df["no2_mean"].diff()

Create CSV File including differences:

In [31]:
df.to_csv("rq_2_berlin_houspa_pollution_2023_2025_diff.csv", index=False)